In [6]:
import os
import glob
import shutil
from pathlib import Path

# ----------------------------
# User settings
# ----------------------------
home_dir  = "/Volumes/BACHI-LAB/MRI_DATA/BIDS" 
dest_root = "/Volumes/synapse/projects/SocialSpace/Projects/SNT-fmri_CUD/Data/Scans/fmriprep"

def ensure_parent(p: Path):
    p.parent.mkdir(parents=True, exist_ok=True)

def copy_file(src: Path, dst: Path, *, dry_run=False):
    ensure_parent(dst)
    if dry_run:
        print(f"[DRY] copy: {src} -> {dst}")
        return
    shutil.copy2(src, dst)  # preserves mtime/metadata
    print(f"[OK ] copy: {src} -> {dst}")

def warn_if_odd(files, label):
    if len(files) > 2:
        print(f"[WARN] Found {len(files)} {label} files (expected ~<=2). Sizes:")
        for f in files:
            try:
                sz = f.stat().st_size
                print(f"       {f.name}  {bytes_to_gb(sz):.2f} GB  ({bytes_to_mb(sz):.0f} MB)")
            except FileNotFoundError:
                print(f"       {f.name}  (missing?)")

def size_sanity_checks(social_files, rest_files):
    for f in social_files:
        sz_gb = bytes_to_gb(f.stat().st_size)
        if sz_gb < SOCIALNAV_MIN_GB:
            print(f"[WARN] socialnav looks small: {f}  ({sz_gb:.2f} GB; expected >~ {SOCIALNAV_MIN_GB} GB)")

    for f in rest_files:
        sz_mb = bytes_to_mb(f.stat().st_size)
        if abs(sz_mb - REST_TARGET_MB) > REST_TOL_MB:
            print(f"[WARN] rest size atypical: {f}  ({sz_mb:.0f} MB; expected ~{REST_TARGET_MB}±{REST_TOL_MB} MB)")

def copy_selected_bolds(home_dir, dest_root, *, dry_run=False, only_subject=None):
    home_dir  = Path(home_dir).expanduser().resolve()
    dest_root = Path(dest_root).expanduser().resolve()
    log_path  = dest_root / "copied_files_by_subject.txt"

    # Expected sizes (rough)
    SOCIALNAV_MIN_GB = 1.8
    REST_TARGET_MB   = 480
    REST_TOL_MB      = 250

    def bytes_to_gb(n): return n / (1024**3)
    def bytes_to_mb(n): return n / (1024**2)

    def _no_echo(paths):
        return [p for p in paths if "echo" not in p.name.lower()]

    def _social_ok(p: Path) -> bool:
        return bytes_to_gb(p.stat().st_size) >= SOCIALNAV_MIN_GB

    def _rest_ok(p: Path) -> bool:
        sz = bytes_to_mb(p.stat().st_size)
        return abs(sz - REST_TARGET_MB) <= REST_TOL_MB

    def _choose_run(files, *, label, size_ok_fn):
        files = list(files)
        if not files:
            return None

        run02 = [f for f in files if "run-02" in f.name]
        run01 = [f for f in files if "run-01" in f.name]

        # prefer run-02 if both exist and size looks ok
        if run02 and run01:
            cand = max(run02, key=lambda p: p.stat().st_size)
            if size_ok_fn(cand):
                return cand
            print(f"[WARN] {label}: run-02 exists but size looks off -> {cand.name}; "
                  "falling back to largest match.")
            return max(files, key=lambda p: p.stat().st_size)

        if run02:
            return max(run02, key=lambda p: p.stat().st_size)
        if run01:
            return max(run01, key=lambda p: p.stat().st_size)
        return max(files, key=lambda p: p.stat().st_size)

    def _companion_json(p_nii: Path) -> Path:
        # .nii.gz -> .json (strip both suffixes)
        if p_nii.name.endswith(".nii.gz"):
            return p_nii.with_name(p_nii.name[:-7] + ".json")
        if p_nii.suffix == ".nii":
            return p_nii.with_suffix(".json")
        return p_nii.with_suffix(".json")

    def _dest_sub_label(src_sub_label: str) -> str:
        """
        Convert destination subject folder label:
          sub-P18001 -> sub-18001
          sub-18001  -> sub-18001
        """
        if src_sub_label.startswith("sub-P"):
            return "sub-" + src_sub_label[len("sub-P"):]
        return src_sub_label

    def _append_subject_log(sub_key: str, entries: list[str]):
        if dry_run:
            print(f"[DRY] would append log for {sub_key}: {log_path}")
            return
        log_path.parent.mkdir(parents=True, exist_ok=True)
        with log_path.open("a", encoding="utf-8") as f:
            f.write(sub_key + "\n")
            for e in entries:
                f.write(f"  {e}\n")
            f.write("\n")

    def _copy_if_needed(src: Path, dst: Path, entries: list[str], *, label=None) -> bool:
        """
        Copy src -> dst only if dst does NOT exist.
        Returns True if copied, False if skipped.
        """
        if dst.exists():
            entries.append(f"SKIP   {dst.name} (already exists)")
            return False
        if not src.exists():
            entries.append(f"WARN   missing source: {src}")
            return False
        copy_file(src, dst, dry_run=dry_run)
        entries.append(f"COPY   {src} -> {dst}")
        return True

    # Build list of subjects (func dirs)
    if only_subject is None:
        func_dirs = sorted(Path(p) for p in glob.glob(str(home_dir / "sub-*" / "func")))
    else:
        sub = only_subject if str(only_subject).startswith("sub-") else f"sub-{only_subject}"
        func_dirs = [home_dir / sub / "func"]

    if not func_dirs or not func_dirs[0].exists():
        print(f"[WARN] No func dirs found for: {only_subject if only_subject else 'sub-*/func/'}")
        return

    missing_social = []
    missing_rest   = []
    missing_anat   = []
    missing_fmap   = []

    for func_dir in func_dirs:
        if not func_dir.exists():
            continue

        sub_dir         = func_dir.parent
        rel_sub_dir_src = sub_dir.relative_to(home_dir)  # e.g., sub-P18001
        src_sub_label   = rel_sub_dir_src.name           # e.g., sub-P18001
        dst_sub_label   = _dest_sub_label(src_sub_label) # e.g., sub-18001

        # Destination subject dir uses dst_sub_label (P removed)
        dest_sub_dir  = dest_root / dst_sub_label
        dest_func_dir = dest_sub_dir / "func"
        dest_anat_dir = dest_sub_dir / "anat"
        dest_fmap_dir = dest_sub_dir / "fmap"

        # FUNC completeness check (new naming convention)
        func_complete = all([
            (dest_func_dir / f"{dst_sub_label}_task-socialnav_bold.nii.gz").exists(),
            (dest_func_dir / f"{dst_sub_label}_task-socialnav_bold.json").exists(),
            (dest_func_dir / f"{dst_sub_label}_task-rest_bold.nii.gz").exists(),
            (dest_func_dir / f"{dst_sub_label}_task-rest_bold.json").exists(),
        ])
        if func_complete:
            print(f"[SKIP] {dst_sub_label}: func already complete (socialnav/rest nii+json).")
            continue

        entries = []  # log entries for THIS subject; flushed after subject completes

        # -----------------------------
        # FUNC: choose social/rest bolds
        # -----------------------------
        social_all = sorted(set(
            Path(p) for p in (
                glob.glob(str(func_dir / "sub-*socialnav_bold.nii.gz")) +
                glob.glob(str(func_dir / "sub-*socialnav_*_bold.nii.gz"))
            )
        ))
        rest_all = sorted(set(
            Path(p) for p in (
                glob.glob(str(func_dir / "sub-*rest_*_bold.nii.gz")) +
                glob.glob(str(func_dir / "sub-*rest_bold.nii.gz")) +
                glob.glob(str(func_dir / "sub-*_rest_bold.nii.gz")) +
                glob.glob(str(func_dir / "sub-*_task-rest_bold.nii.gz"))
            )
        ))
        social_all = _no_echo(social_all)
        rest_all   = _no_echo(rest_all)

        social = _choose_run(social_all, label=f"{dst_sub_label} socialnav", size_ok_fn=_social_ok)
        rest   = _choose_run(rest_all,   label=f"{dst_sub_label} rest",     size_ok_fn=_rest_ok)

        # If no socialnav, skip subject entirely (as before)
        if social is None:
            missing_social.append(dst_sub_label)
            print(f"[WARN] {dst_sub_label}: no socialnav bold found in {func_dir} -> skipping subject")
            continue

        if not _social_ok(social):
            print(f"[WARN] {dst_sub_label}: chosen socialnav looks small: {social.name} "
                  f"({bytes_to_gb(social.stat().st_size):.2f} GB; expected >~ {SOCIALNAV_MIN_GB} GB)")

        if rest is None:
            missing_rest.append(dst_sub_label)
            print(f"[WARN] {dst_sub_label}: no rest bold found in {func_dir}")
        elif not _rest_ok(rest):
            print(f"[WARN] {dst_sub_label}: chosen rest size atypical: {rest.name} "
                  f"({bytes_to_mb(rest.stat().st_size):.0f} MB; expected ~{REST_TARGET_MB}±{REST_TOL_MB} MB)")

        # Ensure dest dirs exist (only when needed)
        if not dry_run:
            dest_func_dir.mkdir(parents=True, exist_ok=True)

        # Copy socialnav (task-..._bold); prepend dst_sub_label; skip per-file if already exists
        social_dst_nii  = dest_func_dir / f"{dst_sub_label}_task-socialnav_bold.nii.gz"
        social_dst_json = dest_func_dir / f"{dst_sub_label}_task-socialnav_bold.json"
        _copy_if_needed(social, social_dst_nii, entries)
        _copy_if_needed(_companion_json(social), social_dst_json, entries)

        # Copy rest (task-..._bold); prepend dst_sub_label; skip per-file if already exists
        if rest is not None:
            rest_dst_nii  = dest_func_dir / f"{dst_sub_label}_task-rest_bold.nii.gz"
            rest_dst_json = dest_func_dir / f"{dst_sub_label}_task-rest_bold.json"
            _copy_if_needed(rest, rest_dst_nii, entries)
            _copy_if_needed(_companion_json(rest), rest_dst_json, entries)

        # -----------------------------
        # ANAT: copy T1w; prepend dst_sub_label (do NOT strip)
        # -----------------------------
        anat_src_dir = sub_dir / "anat"
        if anat_src_dir.exists():
            for suffix in ("T1w.nii.gz", "T1w.json"):
                src = anat_src_dir / f"{src_sub_label}_{suffix}"
                dst = dest_anat_dir / f"{dst_sub_label}_{suffix}"
                if not dry_run:
                    dest_anat_dir.mkdir(parents=True, exist_ok=True)
                ok = _copy_if_needed(src, dst, entries)
                if not ok and not dst.exists() and not src.exists():
                    missing_anat.append(dst_sub_label)
        else:
            missing_anat.append(dst_sub_label)
            entries.append(f"WARN   missing anat dir: {anat_src_dir}")

        # -----------------------------
        # FMAP: copy 4 files; prepend dst_sub_label (do NOT strip)
        # -----------------------------
        fmap_src_dir = sub_dir / "fmap"
        if fmap_src_dir.exists():
            fmap_files = [
                "dir-AP_epi.nii.gz", "dir-AP_epi.json",
                "dir-PA_epi.nii.gz", "dir-PA_epi.json",
            ]
            if not dry_run:
                dest_fmap_dir.mkdir(parents=True, exist_ok=True)
            for fn in fmap_files:
                src = fmap_src_dir / f"{src_sub_label}_{fn}"
                dst = dest_fmap_dir / f"{dst_sub_label}_{fn}"
                ok = _copy_if_needed(src, dst, entries)
                if not ok and not dst.exists() and not src.exists():
                    missing_fmap.append(dst_sub_label)
        else:
            missing_fmap.append(dst_sub_label)
            entries.append(f"WARN   missing fmap dir: {fmap_src_dir}")

        # Flush log after subject completes (log key uses destination label)
        _append_subject_log(dst_sub_label, entries)
        print(f"[OK ] {dst_sub_label}: completed; appended {len(entries)} log entries")

    if missing_social:
        print(f"[SUMMARY WARN] Skipped (missing socialnav) for {len(missing_social)} subjects: {missing_social[:10]}"
              + (" ..." if len(missing_social) > 10 else ""))
    if missing_rest:
        print(f"[SUMMARY WARN] Missing rest for {len(missing_rest)} subjects: {missing_rest[:10]}"
              + (" ..." if len(missing_rest) > 10 else ""))
    if missing_anat:
        uniq = sorted(set(missing_anat))
        print(f"[SUMMARY WARN] Missing anat pieces for {len(uniq)} subjects: {uniq[:10]}"
              + (" ..." if len(uniq) > 10 else ""))
    if missing_fmap:
        uniq = sorted(set(missing_fmap))
        print(f"[SUMMARY WARN] Missing fmap pieces for {len(uniq)} subjects: {uniq[:10]}"
              + (" ..." if len(uniq) > 10 else ""))
    if dry_run:
        print("[NOTE] dry_run=True: nothing was actually copied or written.")

copy_selected_bolds(
    home_dir=home_dir,
    dest_root=dest_root,
    only_subject=None,
    dry_run=False
)

[SKIP] sub-18001: func already complete (socialnav/rest nii+json).
[SKIP] sub-18002: func already complete (socialnav/rest nii+json).
[SKIP] sub-18003: func already complete (socialnav/rest nii+json).
[SKIP] sub-18004: func already complete (socialnav/rest nii+json).
[SKIP] sub-18005: func already complete (socialnav/rest nii+json).
[SKIP] sub-18006: func already complete (socialnav/rest nii+json).
[SKIP] sub-18007: func already complete (socialnav/rest nii+json).
[SKIP] sub-18009: func already complete (socialnav/rest nii+json).
[SKIP] sub-18010: func already complete (socialnav/rest nii+json).
[SKIP] sub-18011: func already complete (socialnav/rest nii+json).
[SKIP] sub-18013: func already complete (socialnav/rest nii+json).
[SKIP] sub-18015: func already complete (socialnav/rest nii+json).
[SKIP] sub-18017: func already complete (socialnav/rest nii+json).
[SKIP] sub-18018: func already complete (socialnav/rest nii+json).
[SKIP] sub-19002: func already complete (socialnav/rest nii+js